# 🏙️ DubaiNest AI — LangChain RAG Pipeline (OpenAI / GPT-4o)
**AI Accelerator Bootcamp — Day 1 & 2**

### What this notebook does:
1. ✅ Installs dependencies
2. ✅ Mounts Google Drive and loads your 3 data files
3. ✅ Chunks documents using LangChain **TextSplitter**
4. ✅ Embeds chunks using LangChain **OpenAIEmbeddings**
5. ✅ Stores vectors in **Chroma** via LangChain VectorStore
6. ✅ Builds a **RetrievalQA chain** (LangChain handles retrieval + prompt + LLM)
7. ✅ Exposes a live API via Flask + ngrok

---
> 🔄 **Upgrade from scratch RAG:** We now use LangChain's built-in pipeline  
> instead of manually wiring embeddings, retrieval and prompts together.  
> Less code. More power. Industry standard.

**Run each cell in order. Every line is commented so you know what it does.**

## CELL 1 — Install all dependencies

In [ ]:
# Install all LangChain packages explicitly with compatible versions
# Pinning versions avoids Colab auto-installing mismatched releases

!pip install -q \
  langchain==0.3.25 \
  langchain-core==0.3.58 \
  langchain-openai==0.3.16 \
  langchain-community==0.3.24 \
  langchain-chroma==0.2.4 \
  langchain-text-splitters==0.3.8 \
  chromadb \
  openai \
  flask flask-cors pyngrok \
  pandas tiktoken

print('✅ All dependencies installed!')

## CELL 2 — Mount Google Drive
Connects Colab to your Drive so we can read the 3 data files.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# ⚠️ Change this to the folder where you saved your 3 data files
DATA_FOLDER = '/content/drive/MyDrive/dubainest_ai/data'

print(f'✅ Drive mounted. Data folder: {DATA_FOLDER}')

## CELL 3 — Load documents with LangChain Loaders
LangChain provides **Document Loaders** that read files and wrap content  
into `Document` objects (text + metadata). We use:
- `CSVLoader` for area_guide.csv
- `TextLoader` for rera_rules.txt and cost_logic.txt

In [ ]:
import os
from langchain_community.document_loaders import CSVLoader, TextLoader

# ── Load area_guide.csv ──────────────────────────────────────────
# CSVLoader turns each row into a separate Document automatically
csv_loader = CSVLoader(
    file_path=os.path.join(DATA_FOLDER, 'area_guide.csv'),
    encoding='utf-8'
)
area_docs = csv_loader.load()
print(f'✅ {len(area_docs)} documents loaded from area_guide.csv')

# ── Load rera_rules.txt ──────────────────────────────────────────
rera_loader = TextLoader(os.path.join(DATA_FOLDER, 'rera_rules.txt'), encoding='utf-8')
rera_docs = rera_loader.load()
print(f'✅ {len(rera_docs)} document loaded from rera_rules.txt')

# ── Load cost_logic.txt ──────────────────────────────────────────
cost_loader = TextLoader(os.path.join(DATA_FOLDER, 'cost_logic.txt'), encoding='utf-8')
cost_docs = cost_loader.load()
print(f'✅ {len(cost_docs)} document loaded from cost_logic.txt')

# ── Combine all documents ────────────────────────────────────────
raw_docs = area_docs + rera_docs + cost_docs
print(f'\n📄 Total raw documents: {len(raw_docs)}')

## CELL 4 — Split documents into chunks with LangChain TextSplitter
Long documents need to be split into smaller chunks before embedding.  
`RecursiveCharacterTextSplitter` splits on paragraph → sentence → word boundaries  
so chunks stay semantically meaningful.

- `chunk_size=500`   → max characters per chunk  
- `chunk_overlap=50` → overlap between chunks so context isn't lost at boundaries

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # max characters per chunk
    chunk_overlap=50,     # overlap to preserve context at boundaries
    separators=['===', '\n\n', '\n', ' ', '']  # try these dividers in order
)

# Split all documents into chunks
all_chunks = splitter.split_documents(raw_docs)

print(f'✅ {len(raw_docs)} raw documents → {len(all_chunks)} chunks after splitting')
print(f'\nSample chunk:')
print(all_chunks[0].page_content[:300])
print(f'\nMetadata: {all_chunks[0].metadata}')

## CELL 5 — Embed chunks and store in Chroma VectorStore
LangChain's `OpenAIEmbeddings` converts text to vectors using OpenAI's  
`text-embedding-3-small` model. `Chroma.from_documents()` embeds and  
stores everything in one line — no manual loop needed.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# ✅ Load OpenAI API key securely from Colab Secrets
# How to set it up (one time only):
#   1. Click the 🔑 key icon in the left sidebar of Colab
#   2. Click "Add new secret"
#   3. Name:  OPENAI_API_KEY
#   4. Value: your actual OpenAI key (sk-...)
#   5. Toggle ON "Notebook access"
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
print('✅ OpenAI API key loaded from Colab Secrets')

# OpenAI Embeddings via LangChain
# text-embedding-3-small is fast and cost-effective
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key=OPENAI_API_KEY
)

# Build Chroma VectorStore from documents in one call
# LangChain handles the embedding loop internally
print('Embedding chunks and storing in Chroma... (may take ~30 seconds)')

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name='dubainest'
)

print(f'✅ {len(all_chunks)} chunks embedded and stored in Chroma VectorStore!')

## CELL 6 — Create a Retriever
A **Retriever** is a LangChain object that wraps the VectorStore  
and returns the most relevant chunks for any query.  
`search_type='similarity'` + `k=4` → return top 4 similar chunks.

In [ ]:
# Convert VectorStore into a retriever
retriever = vectorstore.as_retriever(
    search_type='similarity',  # cosine similarity search
    search_kwargs={'k': 4}     # return top 4 chunks
)

# Test the retriever
test_query = 'What is the average rent for a 1 bedroom in JVC?'
print(f'Query: "{test_query}"\n')

retrieved_docs = retriever.invoke(test_query)
for i, doc in enumerate(retrieved_docs):
    print(f'--- Chunk {i+1} (source: {doc.metadata.get("source", "unknown")}) ---')
    print(doc.page_content[:250] + '...' if len(doc.page_content) > 250 else doc.page_content)
    print()

## CELL 7 — Build the LangChain RAG Chain
This is the key upgrade from scratch RAG.  
LangChain's `RetrievalQA` chain wires together:

```
User Question → Retriever → Prompt Template → LLM → Answer
```

We use a **custom PromptTemplate** to inject the DubaiNest system rules  
and tell Claude to answer only from context.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── LLM ──────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0,
    max_tokens=800,
    openai_api_key=OPENAI_API_KEY
)

# ── format_docs — converts List[Document] → plain string ─────────
def format_docs(docs):
    return '\n\n---\n\n'.join(doc.page_content for doc in docs)

# ── Prompt Template ───────────────────────────────────────────────
PROMPT_TEMPLATE = """
You are DubaiNest AI, a helpful Dubai real estate assistant.
You help users with rental prices, RERA tenant laws, move-in cost
calculations, and area comparisons in Dubai.

Rules:
- Answer ONLY using the CONTEXT provided below. Do not use outside knowledge.
- If the answer is not in the context, say: "I don't have that in my
  knowledge base. Please check dubailand.gov.ae or a licensed agent."
- Always quote prices in AED.
- Be concise. Use bullet points for comparisons and lists.
- Never give legal advice. For legal matters refer users to RDSC.

CONTEXT:
{context}

USER QUESTION:
{question}

Answer based only on the context above.
"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=['context', 'question']
)

# ── RAG Chain ─────────────────────────────────────────────────────
rag_chain = (
    {
        'context':  retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ── ask_dubainest — main function ─────────────────────────────────
def ask_dubainest(user_question):
    return rag_chain.invoke(user_question)

# ── Quick test ────────────────────────────────────────────────────
test_questions = [
    'What is the average rent for a 1 bedroom in JVC?',
    'Can my landlord increase rent by 20 percent?',
    'What is the total move-in cost for a flat at AED 90,000 per year?',
    'Which areas have metro access and 1BR rent under AED 80,000?',
    'Compare JVC vs Sports City for a young professional.',
]

for q in test_questions:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'{"="*60}')
    answer = ask_dubainest(q)
    print(f'A: {answer}')

## CELL 8 — Test the full LangChain RAG pipeline

In [ ]:
test_questions = [
    'What is the average rent for a 1 bedroom in JVC?',
    'Can my landlord increase rent by 20 percent?',
    'What is the total move-in cost for a flat at AED 90,000 per year?',
    'Which areas have metro access and 1BR rent under AED 80,000?',
    'Compare JVC vs Sports City for a young professional.',
]

for q in test_questions:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'{"="*60}')
    answer = ask_dubainest(q)
    print(f'A: {answer}')

## CELL 9 — Export your data files from Colab
Download your 3 knowledge base files — you'll upload them to HuggingFace.

In [ ]:
from google.colab import files
import os

# Download all 3 knowledge base files to your computer
# You will upload these into the data/ folder on HuggingFace

data_files = [
    os.path.join(DATA_FOLDER, 'area_guide.csv'),
    os.path.join(DATA_FOLDER, 'rera_rules.txt'),
    os.path.join(DATA_FOLDER, 'cost_logic.txt'),
]

for f in data_files:
    if os.path.exists(f):
        files.download(f)
        print(f'✅ Downloaded: {os.path.basename(f)}')
    else:
        print(f'⚠️  Not found: {f} — check your DATA_FOLDER path')

print('\n📦 Upload these 3 files into the data/ folder on HuggingFace Space')

## CELL 10 — Deploy to HuggingFace Spaces (Free, Permanent URL)

### Why HuggingFace Spaces?
- ✅ **Free** — no credit card, no signup cost
- ✅ **Permanent URL** — works 24/7, no ngrok needed
- ✅ **Auto-rebuild** — push files → Space rebuilds automatically
- ✅ **Secure secrets** — API key stored safely, never in code

---

### Step-by-step deployment

**Step 1 — Create a free account**
Go to [huggingface.co](https://huggingface.co) → Sign up (free)

**Step 2 — Create a new Space**
- Click your profile → **New Space**
- Space name: `dubainest-ai`
- SDK: **Docker**
- Visibility: **Public**
- Click **Create Space**

**Step 3 — Upload these 4 files** (drag & drop in the Files tab)
```
app.py
requirements.txt
Dockerfile
README.md
```

**Step 4 — Create the data folder and upload your 3 files**
- Click **Add file** → create folder `data`
- Upload: `area_guide.csv`, `rera_rules.txt`, `cost_logic.txt`

**Step 5 — Add your OpenAI API key as a Secret**
- Go to Space **Settings** → **Variables and Secrets**
- Click **New Secret**
- Name: `OPENAI_API_KEY`
- Value: your OpenAI key (`sk-...`)
- Click **Save**

**Step 6 — Wait for build (~3–5 minutes)**
- HuggingFace builds the Docker container automatically
- Watch the **Build logs** tab for progress
- When you see `✅ RAG chain ready!` → your API is live!

**Step 7 — Get your permanent URL**
```
https://YOUR-USERNAME-dubainest-ai.hf.space
```
Example: `https://nipun-dubainest-ai.hf.space`

**Step 8 — Test it**
```
GET  https://YOUR-USERNAME-dubainest-ai.hf.space/health
POST https://YOUR-USERNAME-dubainest-ai.hf.space/chat
     Body: { "question": "What is rent in JVC?" }
```

## CELL 11 — Test your deployed HuggingFace API

In [ ]:
import requests as req

# ⚠️ Replace with your actual HuggingFace Space URL
HF_URL = 'https://YOUR-USERNAME-dubainest-ai.hf.space'

# ── Health check ─────────────────────────────────────────────────
print('Testing health endpoint...')
health = req.get(f'{HF_URL}/health').json()
print('Health:', health)

# ── Chat test ────────────────────────────────────────────────────
test_questions = [
    'What is the average rent for a 1 bedroom in JVC?',
    'Can my landlord increase rent by 20 percent?',
]

for q in test_questions:
    r = req.post(f'{HF_URL}/chat', json={'question': q})
    data = r.json()
    print(f'\nQ: {q}')
    print(f'A: {data.get("answer", data.get("error"))}')

print(f'\n✅ API is live at: {HF_URL}')
print(f'📋 Copy this URL → paste into DubaiNest_AI_Chat.html')

---
## ✅ Full Stack Complete!

```
Google Colab (one-time build)          HuggingFace Spaces (permanent)
─────────────────────────────          ──────────────────────────────
Load CSV + TXT files                   app.py  ← Flask API
Split into 90 chunks          →        Dockerfile
Embed with OpenAIEmbeddings            requirements.txt
Build LangChain RAG chain              data/ folder (3 files)
Test with questions                    OPENAI_API_KEY (secret)
Export data files             →        
                                       Permanent URL 🌐
                                       https://you-dubainest-ai.hf.space
                                              ↓
                                       DubaiNest_AI_Chat.html
                                       (paste URL → Connect → Chat)
```

### 🔑 Secrets needed
| Where | Secret Name | Value |
|---|---|---|
| Colab Secrets | `OPENAI_API_KEY` | `sk-...` from platform.openai.com |
| HuggingFace Space Secrets | `OPENAI_API_KEY` | same key |

### 📁 Files uploaded to HuggingFace
```
dubainest-ai/
├── app.py               ← Flask + LangChain RAG
├── Dockerfile           ← Docker build config  
├── requirements.txt     ← Python packages
├── README.md            ← Space description
└── data/
    ├── area_guide.csv   ← Dubai area rental data
    ├── rera_rules.txt   ← RERA tenant law rules
    └── cost_logic.txt   ← Move-in cost calculations
```